# Monte Carlo Robustness Example

This notebook runs a baseline backtest and then evaluates robustness using the `trade_lab.monte_carlo` module.

## 1) Imports and path setup

If TradeLab is not installed as a package, the next cell adds `../src` to `sys.path`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'examples' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.backtesting import BacktestEngine
from trade_lab.indicators import EMA, RSI
from trade_lab.strategies import StandardStrategy
from trade_lab.monte_carlo import (
    BlockBootstrap,
    CircularBlockBootstrap,
    GBMSimulator,
    ReturnShuffler,
    MonteCarloRunner,
    MonteCarloAnalysis,
)

## 2) Define strategy and fetch data

In [ ]:
strategy = StandardStrategy(
    indicators=[
        (EMA(period=20), 1.0),
        (EMA(period=50), -1.0),
        (RSI(period=14), 0.30),
    ],
    allow_long=True,
    allow_short=True,
    entry_threshold=0.20,
    exit_threshold=0.05,
)

engine = BacktestEngine(
    strategy=strategy,
    ticker='SPY',
    start='2018-01-01',
    end='2025-01-01',
    initial_capital=100_000.0,
    commission=0.001,
    slippage=0.0005,
)

original_df = engine.fetch_data()
baseline = engine.run_on(original_df)

print('Rows:', len(original_df))
print('Baseline sharpe:', round(baseline.metrics['sharpe_ratio'], 4))
print('Baseline total return:', round(baseline.metrics['total_return'], 4))
print('Baseline max drawdown:', round(baseline.metrics['max_drawdown'], 4))

## 3) Run Monte Carlo with Stationary Block Bootstrap

In [ ]:
metrics = ['total_return', 'sharpe_ratio', 'max_drawdown', 'win_rate']

runner = MonteCarloRunner(
    engine=engine,
    generator=BlockBootstrap(block_size=20, seed=42),
    n_simulations=300,
    metrics=metrics,
    verbose=True,
)

mc_result = runner.run(original_df)
analysis = MonteCarloAnalysis(mc_result)
summary = analysis.summary()
summary.round(4)

In [ ]:
for metric in metrics:
    lo, hi = analysis.confidence_interval(metric, lower=5, upper=95)
    base_value = baseline.metrics[metric]
    pct = analysis.percentile_of(metric, base_value)
    print(f"{metric:>14} | baseline={base_value: .4f} | CI[5,95]=({lo: .4f}, {hi: .4f}) | percentile={pct: .2f}%")

## 4) Plot simulated distributions vs baseline

In [ ]:
dist = analysis.distributions()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, metric in zip(axes, metrics):
    values = dist[metric]
    finite = values[np.isfinite(values)]
    ax.hist(finite, bins=30, alpha=0.8)
    ax.axvline(baseline.metrics[metric], color='red', linestyle='--', linewidth=2, label='baseline')
    ax.set_title(metric)
    ax.legend()

plt.tight_layout()
plt.show()

## 5) Optional: Compare generators quickly

In [ ]:
generators = {
    'BlockBootstrap': BlockBootstrap(block_size=20, seed=42),
    'CircularBlockBootstrap': CircularBlockBootstrap(block_size=20, seed=42),
    'ReturnShuffler': ReturnShuffler(seed=42),
    'GBM': GBMSimulator(seed=42),
}

rows = []
for name, gen in generators.items():
    r = MonteCarloRunner(engine=engine, generator=gen, n_simulations=120, metrics=['sharpe_ratio'], verbose=False)
    a = MonteCarloAnalysis(r.run(original_df))
    lo, hi = a.confidence_interval('sharpe_ratio', 5, 95)
    pct = a.percentile_of('sharpe_ratio', baseline.metrics['sharpe_ratio'])
    rows.append((name, lo, hi, pct))

import pandas as pd
pd.DataFrame(rows, columns=['generator', 'sharpe_p5', 'sharpe_p95', 'baseline_sharpe_percentile']).round(4)